# Seq2Seq Translation (de → en) with Transformers

This notebook implements a translation pipeline using a Seq2Seq transformer model from the Hugging Face Transformers library. We evaluate its performance using metrics like BLEU and ROUGE.

In [ ]:
# Install required packages
!pip install -U transformers datasets evaluate sacrebleu rouge_score sentencepiece torch

### 1) Imports and basic setup

In [ ]:
import os
import random
import numpy as np
from datasets import load_dataset
import evaluate
import nltk
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

# reproducibility
RANDOM_SEED = 42
set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print("PyTorch device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

### 2) Load a small subset of the dataset (IWSLT2017 de-en)

In [ ]:
# Load a parallel dataset for German -> English
raw_datasets = load_dataset("iwslt2017", "de-en")
print(raw_datasets)

# We'll take small subsets for faster demo runs.
train_ds = raw_datasets["train"].select(range(2000))
valid_ds = raw_datasets["validation"].select(range(500))
test_ds = raw_datasets["test"].select(range(500))

print("Example:", train_ds[0])

### 3) Model & tokenizer
Using the pre-trained Helsinki opus model for de→en translation: `Helsinki-NLP/opus-mt-de-en`

In [ ]:
model_name = "Helsinki-NLP/opus-mt-de-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

print("Tokenizer vocab size:", len(tokenizer))
# print("Model config:", model.config)

### 4) Preprocessing function

In [ ]:
SRC_LANG = "de"
TGT_LANG = "en"
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    inputs = [t[SRC_LANG] for t in examples["translation"]]
    targets = [t[TGT_LANG] for t in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_valid = valid_ds.map(preprocess_function, batched=True, remove_columns=valid_ds.column_names)
tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=test_ds.column_names)

### 5) Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

### 6) Metrics Setup

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]
    return preds, labels

def exact_match_and_token_acc(preds, labels):
    exact_matches = 0
    total_tokens = 0
    matched_tokens = 0
    for p, l in zip(preds, labels):
        if p == l:
            exact_matches += 1
        ptoks = p.split()
        ltoks = l.split()
        L = max(len(ptoks), len(ltoks))
        for i in range(L):
            if i < len(ptoks) and i < len(ltoks) and ptoks[i] == ltoks[i]:
                matched_tokens += 1
            total_tokens += 1
    return {
        "exact_match": exact_matches / len(preds),
        "token_accuracy": matched_tokens / total_tokens if total_tokens > 0 else 0.0,
    }

### 7) Compute Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    
    bleu_result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    other = exact_match_and_token_acc(decoded_preds, decoded_labels)
    
    return {
        "bleu": bleu_result["bleu"],
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "exact_match": other["exact_match"],
        "token_accuracy": other["token_accuracy"],
    }

### 8) Trainer Setup

In [ ]:
output_dir = "./de-en-opus-demo"
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=200,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    seed=RANDOM_SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### 9) Training (Optional)
Set `do_train = True` to run training. By default, we use the pre-trained model for evaluation.

In [ ]:
do_train = False
if do_train:
    trainer.train()
else:
    print("Skipping training. Using pre-trained model for evaluation.")

### 10) Evaluation and Results

In [ ]:
print("Running evaluation on test set...")
eval_result = trainer.evaluate(eval_dataset=tokenized_test, max_length=128, num_beams=4)
print("Evaluation results:", eval_result)

small_test = tokenized_test.select(range(5))
preds = trainer.predict(small_test, max_length=128, num_beams=4)
decoded_preds = tokenizer.batch_decode(preds.predictions, skip_special_tokens=True)
labels = np.where(preds.label_ids != -100, preds.label_ids, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

for i, (pred, ref) in enumerate(zip(decoded_preds, decoded_labels)):
    print(f"Example {i+1}:")
    print(f"  Reference: {ref}")
    print(f"  Prediction: {pred}\n")